# 🧠 OmniFile AI Processor — Interactive Debugger (Gradio)

## مشروع OmniFile_Processor
نظام متكامل يجمع بين قوة المعالجة اللغوية (NLP) وتقنيات التعرف الضوئي على الحروف (OCR) مع التركيز على المحتوى العربي.

---

### 📋 ما يغطيه هذا النوت بوك:
1. تثبيت المتطلبات (Setup)
2. واجهة Gradio تفاعلية للتجربة والتصحيح
3. **تبويب البحث الذكي (FTS5)** — محرك بحث شخصي في الأرشيف
4. **نظام الكاش (Hash-based Cache)** — تجنب إعادة المعالجة
5. دعم رفع الملفات من الجوال عبر `share=True`

> **ملاحظة:** يعمل على Google Colab وManjaro Linux و أي حاسوب به Python 3.10+

---
## ⚙️ الخلية الأولى: تثبيت المتطلبات (Setup)

In [ ]:
# 1. تنزيل المشروع من GitHub
!git clone https://github.com/DrAbdulmalek/OmniFile_Processor.git
%cd OmniFile_Processor

# 2. تثبيت المكتبات اللازمة
!pip install gradio easyocr transformers torch openpyxl python-docx PyMuPDF rapidfuzz
!pip install ar-corrector arabic-reshaper python-bidi langdetect jiwer

# 3. فحص نوع المعالج
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ الجهاز: {device}")
if device == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

---
## 🔧 الخلية الثانية: واجهة Gradio مع البحث الذكي + نظام الكاش

تحتوي على **3 تبويبات**:
1. **المعالجة** — رفع ملفات ومعالجتها بال OCR/NLP
2. **محرك البحث** — بحث نصي فوري في آلاف المستندات (FTS5)
3. **الإحصائيات** — لوحة معلومات شاملة عن الأرشيف

In [ ]:
import gradio as gr
import os
import torch
from pathlib import Path
import json

# ============================================================
# إعدادات المشروع
# ============================================================
PROJECT_ROOT = Path("/content/OmniFile_Processor")  # Colab
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path(".")  # Local

import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

device = "cuda" if torch.cuda.is_available() else "cpu"

# ============================================================
# استيراد قاعدة البيانات المحسّنة
# ============================================================
from modules.core.database_manager import OmniDatabase

DB_PATH = PROJECT_ROOT / "omni_archive.db"
db = OmniDatabase(str(DB_PATH))
print(f"✅ قاعدة البيانات: {DB_PATH}")

# ============================================================
# دالة المعالجة الرئيسية (مع نظام الكاش)
# ============================================================
def omni_worker(input_file, mode, engine_choice):
    """
    حلقة الوصل بين واجهة المستخدم ومنطق المشروع.
    يتضمن نظام Hash-based Cache: لن يُعالج نفس الملف مرتين.
    
    استبدل المحتوى الداخلي بدوال مشروعك الحقيقية:
      from modules.vision.ocr_engine import OCREngine
      from modules.nlp.translator import Translator
      from modules.nlp.text_classifier import TextClassifier
    """
    if input_file is None:
        return "⚠️ الرجاء رفع ملف للمعالجة", ""
    
    file_path = input_file.name
    filename = os.path.basename(file_path)
    ext = filename.rsplit('.', 1)[-1].lower()
    
    # --- تنظيف الذاكرة ---
    if device == "cuda":
        torch.cuda.empty_cache()
    
    # --- حساب بصمة الملف (Hash) ---
    file_hash = OmniDatabase.calculate_file_hash(file_path)
    cached = db.check_file_exists(file_hash)
    
    if cached:
        result_text = (
            f"⚡ نتيجة من الكاش (تمت معالجته سابقاً)\n"
            f"{'─' * 40}\n"
            f"📂 الملف: {filename}\n"
            f"🏷️ التصنيف: {cached[0]}\n"
            f"📊 نسبة الثقة: {cached[2]:.0%}\n"
            f"{'─' * 40}\n"
            f"📄 النص المستخرج:\n{cached[1][:500]}"
        )
        return result_text, "✅ تم استرجاع النتيجة من الكاش"
    
    try:
        info_lines = [
            f"📂 الملف: {filename}",
            f"📐 الحجم: {os.path.getsize(file_path) / 1024:.1f} KB",
            f"🔧 المحرك: {engine_choice}",
            f"📋 الوضع: {mode}",
            f"💻 الجهاز: {device}",
            f"{'─' * 40}"
        ]
        
        extracted_text = ""
        category = "unknown"
        
        # ===== تصنيف (Classification) =====
        if "\u062a\u0635\u0646\u064a\u0641" in mode:
            info_lines.append("\n🏷️ نتيجة التصنيف:")
            info_lines.append(f"  النوع المتوقع: مستند {ext}")
            category = f"document_{ext}"
        
        # ===== تعرف ضوئي (OCR) =====
        elif "\u062a\u0639\u0631\u0641" in mode or "OCR" in mode:
            info_lines.append(f"\n🔍 نتيجة OCR:")
            info_lines.append(f"  المحرك المختار: {engine_choice}")
            category = f"ocr_{engine_choice.lower().replace(' ', '_')}"
            if engine_choice == "Hybrid (EasyOCR + TrOCR)":
                info_lines.append(f"  💡 Hybrid: EasyOCR للكشف + TrOCR للقراءة")
        
        # ===== ترجمة (Translation) =====
        elif "\u062a\u0631\u062c\u0645" in mode:
            info_lines.append(f"\n🌐 نتيجة الترجمة:")
            category = "translation"
        
        result_text = "\n".join(info_lines)
        
        # --- حفظ في قاعدة البيانات للكاش المستقبلي ---
        extracted_text = result_text  # في الإنتاج: استبدل بالنص الحقيقي من OCR
        db.save_record(
            file_hash=file_hash,
            file_name=filename,
            file_path=file_path,
            category=category,
            text=extracted_text,
            confidence=0.0,  # في الإنتاج: نسبة الثقة من OCR
            ocr_engine=engine_choice
        )
        info_lines.append(f"\n💾 تم حفظ النتيجة في قاعدة البيانات (كاش)")
        result_text = "\n".join(info_lines)
        
        if device == "cuda":
            torch.cuda.empty_cache()
        
        return result_text, ""
    
    except Exception as e:
        return f"❌ خطأ أثناء المعالجة:\n{str(e)}", str(type(e).__name__ + ": " + str(e))


# ============================================================
# دالة البحث الذكي (FTS5)
# ============================================================
def search_in_archive(search_term):
    """
    بحث نصي فوري في جميع الملفات المعالجة.
    يستخدم محرك FTS5 المدمج في SQLite.
    """
    if not search_term or not search_term.strip():
        return "⚠️ الرجاء إدخال كلمة أو جملة للبحث"
    
    results = db.search_text(search_term, limit=30)
    
    if not results:
        return f"لم يتم العثور على نتائج لـ: "{search_term}"\n\n💡 نصائح:\n- جرب كلمات مختلفة\n- استخدم كلمات مفتاحية بدلاً من جمل كاملة\n- تأكد من أن الملفات تم معالجتها في تبويب المعالجة أولاً"
    
    formatted = f"🔍 نتائج البحث عن: "{search_term}" ({len(results)} نتيجة)\n{'═' * 50}\n\n"
    for i, res in enumerate(results, 1):
        formatted += (
            f"📄 [{i}] {res['file_name']}\n"
            f"   🏷️ التصنيف: {res['category']}\n"
            f"   📊 الثقة: {res['confidence_score']:.0%}\n"
            f"   📅 التاريخ: {res['process_date']}\n"
            f"   💬 السياق: ...{res.get('context_snippet', 'N/A')}...\n"
            f"{'─' * 45}\n"
        )
    
    return formatted


# ============================================================
# دالة الإحصائيات
# ============================================================
def show_statistics():
    """
    عرض إحصائيات شاملة عن قاعدة البيانات.
    """
    stats = db.get_statistics()
    
    report = (
        f"📊 لوحة معلومات الأرشيف\n{'═' * 50}\n\n"
        f"📁 إجمالي الملفات المعالجة: {stats['total_files']}\n"
        f"⚡ من الكاش: {stats['cached_files']}\n"
        f"📈 متوسط الثقة: {stats['average_confidence']:.0%}\n"
        f"✏️ التصحيحات اليدوية: {stats['total_corrections']}\n\n"
    )
    
    if stats['categories']:
        report += "🏷️ التصنيفات:\n"
        for cat in stats['categories']:
            report += f"   • {cat['category']}: {cat['count']} ملف\n"
        report += "\n"
    
    if stats['languages']:
        report += "🌍 اللغات:\n"
        for lang in stats['languages']:
            report += f"   • {lang['language']}: {lang['count']} ملف\n"
        report += "\n"
    
    if stats['engines']:
        report += "🔧 المحركات:\n"
        for eng in stats['engines']:
            report += f"   • {eng['ocr_engine']}: {eng['count']} ملف\n"
    
    return report


# ============================================================
# بناء واجهة غراديو مع 3 تبويبات
# ============================================================
with gr.Blocks(
    theme=gr.themes.Soft(),
    title="OmniFile AI Processor",
    css=".container { max-width: 950px; margin: auto; }"
) as demo:
    
    gr.Markdown("# 🧠 OmniFile AI Processor — Interactive Debugger")
    gr.Markdown(
        "نظام متكامل يجمع بين قوة المعالجة اللغوية (NLP) "
        "وتقنيات التعرف الضوئي على الحروف (OCR) مع التركيز على المحتوى العربي."
    )
    
    with gr.Tabs():
        
        # ===== التبويب 1: المعالجة =====
        with gr.Tab("📂 المعالجة"):
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown("### المدخلات")
                    file_input = gr.File(
                        label="ارفع الملف (PDF, Image, Docx)",
                        file_types=[".pdf", ".png", ".jpg", ".jpeg", ".tiff", ".bmp", ".docx", ".txt"]
                    )
                    mode_radio = gr.Radio(
                        ["\u062a\u0635\u0646\u064a\u0641 (Classification)", "\u062a\u0639\u0631\u0641 \u0636\u0648\u0626\u064a (OCR)", "\u062a\u0631\u062c\u0645\u0629 (Translation)"],
                        label="نوع العملية",
                        value="\u062a\u0639\u0631\u0641 \u0636\u0648\u0626\u064a (OCR)"
                    )
                    engine_radio = gr.Radio(
                        ["EasyOCR", "Tesseract", "TrOCR", "PaddleOCR",
                         "Hybrid (EasyOCR + TrOCR)", "Fusion (All Engines)"],
                        label="محرك OCR",
                        value="EasyOCR"
                    )
                    submit_btn = gr.Button("\U0001f680 بدء المعالجة", variant="primary", size="lg")
                
                with gr.Column(scale=1):
                    gr.Markdown("### المخرجات")
                    output_text = gr.Textbox(label="نتيجة المعالجة", lines=15, show_copy_button=True)
                    error_text = gr.Textbox(label="التتبع (Debug)", lines=5)
            
            submit_btn.click(
                fn=omni_worker,
                inputs=[file_input, mode_radio, engine_radio],
                outputs=[output_text, error_text]
            )
        
        # ===== التبويب 2: محرك البحث الذكي =====
        with gr.Tab("\U0001f50d محرك البحث الذكي"):
            gr.Markdown(
                "### البحث النصي الكامل (FTS5)\n"
                "ابحث في محتوى آلاف المستندات بسرعة فائقة. "
                "مثال: \"كسر عنق الفخذ\" أو \"total knee replacement\""
            )
            with gr.Row():
                with gr.Column(scale=3):
                    search_input = gr.Textbox(
                        label="ابحث في أرشيفك",
                        placeholder="أدخل كلمة أو جملة للبحث...",
                        lines=2
                    )
                with gr.Column(scale=1):
                    search_btn = gr.Button("\U0001f50e ابحث الآن", variant="primary", size="lg")
            
            search_output = gr.Textbox(
                label="نتائج البحث والسياق",
                lines=20,
                show_copy_button=True
            )
            
            search_btn.click(fn=search_in_archive, inputs=search_input, outputs=search_output)
            
            gr.Examples(
                examples=[
                    ["كسر"],
                    ["عملية"],
                    ["تقرير"],
                    ["مريض"],
                ],
                inputs=search_input,
                label="أمثلة للبحث السريع"
            )
        
        # ===== التبويب 3: الإحصائيات =====
        with gr.Tab("\U0001f4ca الإحصائيات"):
            gr.Markdown(
                "### لوحة معلومات الأرشيف\n"
                "إحصائيات شاملة عن الملفات المعالجة والتصنيفات والأداء."
            )
            stats_btn = gr.Button("\U0001f4ca عرض الإحصائيات", variant="secondary", size="lg")
            stats_output = gr.Textbox(
                label="الإحصائيات",
                lines=20,
                show_copy_button=True
            )
            
            stats_btn.click(fn=show_statistics, outputs=stats_output)
    
    gr.Markdown("---")
    gr.Markdown(
        "### 💡 مقترحات التطوير\n"
        "1. **منطق هجين (Hybrid):** EasyOCR للكشف + TrOCR للقراءة → دقة أعلى في الوثائق الطبية\n"
        "2. **نظام الكاش (Hash):** لا يُعالج نفس الملف مرتين → سرعة فائقة مع الأرشيفات الكبيرة\n"
        "3. **التصحيح اللغوي:** بناء معجم طبي متخصص لمصطلحات الجراحة العظمية\n"
        "4. **البحث الدلالي:** مستقبلاً استخدام FAISS/ChromaDB للبحث بالمعنى لا بالكلمات"
    )

# ============================================================
# تشغيل الواجهة
# ============================================================

# share=True ← يعطي رابطاً عاماً (gradio.live) يمكنك فتحه من الجوال
# debug=True ← يظهر الأخطاء (Traceback) مباشرة في المتصفح
demo.launch(share=True, debug=True)

---
## 📱 طريقة الاستخدام من الجوال

| الطريقة | الخطوات |
|---------|--------|
| **عبر الحاسوب** | شغّل `share=True` وافتح رابط gradio.live من متصفح الجوال |
| **عبر كولاب** | افتح هذا النوت بوك من Chrome/Safari على الجوال — الواجهة كاملة |

### التشغيل المحلي على Manjaro Linux:
```bash
git clone https://github.com/DrAbdulmalek/OmniFile_Processor
cd OmniFile_Processor
python -m venv venv
source venv/bin/activate
pip install -r requirements.txt
# ثم شغّل الخلية الثانية أعلاه
```

---
## 🔬 مقترحات التطوير المتقدمة

### 1. التصنيف الدلالي (Semantic Classification)
- استخدم **AraBERT** أو **CamelBERT** بدلاً من Keyword Matching
- ستفهم أن "كسر مضاعف" و"تثبيت داخلي" تتبع لقسم "الجراحة العظمية"
- استخدم **نسبة الثقة** بدلاً من تصنيف واحد صارم

### 2. البحث الدلالي (Vector Search)
- مستقبلاً: استخدم **FAISS** أو **ChromaDB** للبحث بالمعنى
- اسأل: "أين الملف الذي يتحدث عن عمليات تبديل مفصل الورك؟"

### 3. معالجة الأداء
- **Batch Processing:** تجميع الصور وإرسالها للـ GPU دفعة واحدة
- **Hash Cache:** ✅ مُنفَّذ — لا يُعالج نفس الملف مرتين

### 4. نظام الوسوم (Tagging)
- بدلاً من مجلد واحد، أضف وسوم متعددة: (طبي، 2026، جامعة)
- ✅ مُنفَّذ في `database_manager.py` عبر حقل `tags`